In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
from torch.utils.data import DataLoader
from src.rwf2000 import RWF2000Dataset
from src.cnn_lstm.cnn_lstm_v3 import CNNLSTMV3
from src.config import DATASET_ROOT
from scripts.common.get_device import get_available_device

In [3]:
device = get_available_device()

train_dataset = RWF2000Dataset(
    root_dir=DATASET_ROOT,
    split="train",
    num_frames=32,
    image_size=224
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

model = CNNLSTMV3().to(device)
model.eval()

# Get one batch
videos, labels = next(iter(train_loader))
videos = videos.to(device)

print("Input shape:", videos.shape)

# Run a forward pass (your print statements inside the model will execute)
with torch.no_grad():
    logits = model(videos)

print("Output logits shape:", logits.shape)

Using cuda:1 with 20.06 GB free
Input shape: torch.Size([4, 32, 3, 224, 224])
features shape: torch.Size([128, 1280, 7, 7])
channel reduce features shape: torch.Size([128, 64, 7, 7])
restore video structure: torch.Size([4, 32, 64, 7, 7])
output sepconvlstm: torch.Size([4, 32, 128, 7, 7])
last hidden: torch.Size([4, 128, 7, 7])
last hidden maxpool: torch.Size([4, 128, 3, 3])
last hidden avg pool: torch.Size([4, 128, 1, 1])
flattened: torch.Size([4, 128])
Output logits shape: torch.Size([4, 2])


In [2]:
from src.cnn_lstm.cnn_lstm_v3 import CNNLSTMV3
import torch

model = CNNLSTMV3()

dummy = torch.zeros(1, 3, 224, 224)

with torch.no_grad():
    out = model.cnn(dummy)

print(out.shape)
print("feature_dim =", out.shape[1])

torch.Size([1, 1280, 7, 7])
feature_dim = 1280


In [9]:
import torch
from torch.utils.data import DataLoader
from src.rwf2000 import RWF2000Dataset
from src.baseline_cnn_lstm import BaselineCNNLSTM
from src.config import DATASET_ROOT

In [10]:
train_dataset = RWF2000Dataset(
    root_dir=DATASET_ROOT,
    split="train",
    num_frames=32,
    image_size=224
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

In [11]:
videos, labels = next(iter(train_loader))

print(videos.shape)
print(labels.shape)
print(labels)

torch.Size([4, 32, 3, 224, 224])
torch.Size([4])
tensor([1, 0, 0, 1])


In [12]:
model = BaselineCNNLSTM(
    hidden_size=256,
    num_layers=1,
    num_classes=2,
    dropout=0.3,
    freeze_cnn=True
)

outputs = model(videos)

print("Input:", videos.shape)
print("Output:", outputs.shape)
print("Labels:", labels.shape)

Input: torch.Size([4, 32, 3, 224, 224])
Output: torch.Size([4, 2])
Labels: torch.Size([4])


In [13]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

loss = criterion(outputs, labels)

print(loss)

tensor(0.6623, grad_fn=<NllLossBackward0>)
